### Install & Import Libraries

In [1]:
!pip install datasets scikit-learn


In [2]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report


In [3]:
dataset = load_dataset("sms_spam")
df = pd.DataFrame(dataset['train'])
df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [4]:
X_train, X_test, y_train, y_test = train_test_split(df['sms'], df['label'], test_size=0.2, random_state=42)


In [5]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_ifidf = tfidf.fit_transform(X_train)
X_test_ifidf = tfidf.transform(X_test)



In [6]:
model = MultinomialNB()
model.fit(X_train_ifidf, y_train)

MultinomialNB()

In [7]:
preds = model.predict(X_test_ifidf)


In [8]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       954
           1       1.00      0.83      0.90       161

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.95      1115
weighted avg       0.98      0.97      0.97      1115



In [10]:
message = ['Congratulations! You won a  free i']
message_tfidf = tfidf.transform(message)
print("Predictions: ", model.predict(message_tfidf))

Predictions:  [1]


### Spam Detection Text Classification: Naive Bayes, BiLSTM, and BERT Approaches

This project performs spam detection on SMS messages using three approaches: Naive Bayes, BiLSTM, and BERT. It demonstrates a progression from a fast, classical baseline (Naive Bayes) to a sequential deep learning model (BiLSTM) and finally a state-of-the-art transformer model (BERT) for accurate text classification. The goal is to classify messages as spam or ham effectively.

In [11]:
!pip install pandas scikit-learn tensorflow transformers


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score


In [20]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label'], test_size=0.2, random_state=42)






In [22]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [23]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [25]:
# Make predictions on test set
y_pred = nb_model.predict(X_test_tfidf)

# Evaluate the model
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Naive Bayes Accuracy: 0.97847533632287

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       1.00      0.84      0.91       149

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Spam detection using a BiLSTM

In [26]:
!pip install tensorflow pandas scikit-learn


In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score



In [31]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

# Fix parsing errors
df = pd.read_csv(url, sep='\t', header=None, names=['label','text'], engine='python', quoting=3)

print(df.head())
print(df.columns)


  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Index(['label', 'text'], dtype='object')


In [32]:


# Map labels to 0 (ham) and 1 (spam)
df['label'] = df['label'].map({'ham':0, 'spam':1})

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)



In [33]:
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)


In [34]:
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

In [42]:
from tensorflow.keras.layers import GlobalMaxPooling1D

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [43]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()
# Train model
early_stop = EarlyStopping(monitor='val_loss', patience=3)
history = model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 82s 160ms/step - accuracy: 0.8605 - loss: 0.3956 - val_accuracy: 0.9641 - val_loss: 0.0871
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 18s 144ms/step - accuracy: 0.9872 - loss: 0.0416 - val_accuracy: 0.9731 - val_loss: 0.0868
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 20s 161ms/step - accuracy: 0.9958 - loss: 0.0218 - val_accuracy: 0.9798 - val_loss: 0.0905
Epoch 4/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 18s 144ms/step - accuracy: 0.9978 - loss: 0.0097 - val_accuracy: 0.9821 - val_loss: 0.0954
Epoch 5/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 19s 153ms/step - accuracy: 0.9993 - loss: 0.0039 - val_accuracy: 0.9753 - val_loss: 0.0955


In [44]:
y_pred = (model.predict(X_test_seq) > 0.5).astype("int32")

# Evaluate
print("BiLSTM Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step
BiLSTM Accuracy: 0.9919282511210762

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       954
           1       0.98      0.96      0.97       161

    accuracy                           0.99      1115
   macro avg       0.99      0.98      0.98      1115
weighted avg       0.99      0.99      0.99      1115

